# 🏦 AIRG Enterprise Demo — Fintech Dual-Agent Governance

**Organization 2** · Two-Agent Governance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/othnielObasi/airg-cookbook/blob/main/use_cases/enterprise_demos/org2_fintech_dual_agents.ipynb)

---

This notebook demonstrates how a fintech organization governs **two AI agents** under a single AIRG tenant:

| Agent | Role | Tools |
|-------|------|-------|
| **trading-agent-v1** | Executes trades, checks portfolios | `execute_trade`, `check_portfolio`, `market_data` |
| **compliance-agent-v1** | Reviews transactions, files reports | `review_transaction`, `file_sar`, `kyc_lookup` |

Both agents share the **same org-level settings** but have independent fingerprint baselines, session histories, and SURGE receipt chains.

### Modules Demonstrated
Settings configuration, evaluate pipeline, output scanning, verification, fingerprinting (per-agent baselines), impact assessment (cross-agent), SURGE v2, traces, chain detection, budget enforcement, escalation.

## 1. Setup

In [ ]:
!pip install -q httpx tabulate

import httpx, json, time, uuid
from tabulate import tabulate

API_BASE = "https://api.airg.nov-tia.com"
API_KEY  = "airg_5YbH98EvjL28TZwKPXLQuoY2Frgk01-O7ofBaIrhK8A"
HEADERS  = {"X-API-Key": API_KEY, "Content-Type": "application/json"}

# Two agents, one org
TRADING_AGENT    = "trading-agent-v1"
COMPLIANCE_AGENT = "compliance-agent-v1"
TRADING_SESSION    = f"trading-{uuid.uuid4().hex[:8]}"
COMPLIANCE_SESSION = f"compliance-{uuid.uuid4().hex[:8]}"

client = httpx.Client(base_url=API_BASE, headers=HEADERS, timeout=30, follow_redirects=True)
print(f"✅ Connected to AIRG — Fintech Org")
print(f"   Trading Agent:    {TRADING_AGENT} (session: {TRADING_SESSION})")
print(f"   Compliance Agent: {COMPLIANCE_AGENT} (session: {COMPLIANCE_SESSION})")

## 2. Org-Level Settings — Shared Configuration

In [ ]:
# View settings (shared across both agents)
r = client.get("/settings")
if r.status_code == 200:
    s = r.json()
    print("=== Fintech Org Settings ===")
    # Show key settings
    keys = ["kill_switch", "modules_enabled", "injection_detector_enabled",
            "pii_scanner_enabled", "budget_enforcer_enabled", "fingerprinting_enabled",
            "impact_assessment_enabled", "surge_v2_enabled", "content_moderator_enabled"]
    for k in keys:
        if k in s:
            print(f"  {k:40s} = {s[k]}")
else:
    print(f"Settings: {r.status_code}")

# Configure budget limits for trading
r = client.patch("/settings", json={
    "budget_max_evals_per_hour": 500,      # Trading generates many evaluations
    "budget_max_evals_per_day": 5000,
    "pii_scanner_enabled": True,            # PII scanning for KYC data
    "fingerprinting_enabled": True,         # Critical for detecting rogue trading
})
if r.status_code == 200:
    print("\n✅ Budget & fingerprinting configured for trading workload")
else:
    print(f"Update: {r.status_code} — {r.text[:200]}")

In [ ]:
# List modules
r = client.get("/settings/modules")
if r.status_code == 200:
    mods = r.json()
    table = [[m["name"], "✅" if m["enabled"] else "❌", "✅" if m["loaded"] else "❌"] for m in mods]
    print(tabulate(table, headers=["Module", "Enabled", "Loaded"], tablefmt="simple"))
else:
    print(f"Modules: {r.status_code}")

## 3. Evaluate — Dual-Agent Governance

In [ ]:
def evaluate(tool, args, agent_id, session_id, allowed_tools=None):
    """Evaluate a tool call and display comprehensive results."""
    payload = {
        "tool": tool,
        "args": args,
        "agent_id": agent_id,
        "session_id": session_id,
        "context": {
            "agent_id": agent_id,
            "session_id": session_id,
            "trace_id": f"trace-{uuid.uuid4().hex[:12]}",
            "allowed_tools": allowed_tools or []
        }
    }
    r = client.post("/actions/evaluate", json=payload)
    d = r.json()
    
    icon = {"allow": "✅", "review": "⚠️", "block": "🚫"}.get(d.get("decision"), "❓")
    print(f"\n{'='*60}")
    print(f"{icon}  {d.get('decision','?').upper()}  |  Risk: {d.get('risk_score', '?')}/100  |  Agent: {agent_id}")
    print(f"   Tool: {tool}")
    print(f"{'='*60}")
    print(f"Explanation: {d.get('explanation', 'N/A')}")
    
    # Trace
    if d.get("execution_trace"):
        print(f"\nExecution Trace:")
        for step in d["execution_trace"]:
            s_icon = {"pass": "✅", "block": "🚫", "review": "⚠️"}.get(step.get("outcome"), "?")
            print(f"  L{step['layer']} {step['name']:20s} {s_icon} {step['outcome']:6s}  risk+{step.get('risk_contribution',0):2d}")
    
    # Modules
    if d.get("chain_pattern"):
        print(f"🔗 Chain: {d['chain_pattern']} — {d.get('chain_description', '')}")
    if d.get("deviation_count", 0) > 0:
        print(f"🔍 Deviations: {d['deviation_count']} — {d.get('deviation_types', [])}")
    if d.get("pii_findings_count", 0) > 0:
        print(f"🛡️ PII findings: {d['pii_findings_count']}")
    if d.get("evidence_hash"):
        print(f"🔒 Evidence: {d['evidence_hash'][:32]}...")
    if d.get("escalation_id"):
        print(f"🚨 Escalated: severity={d.get('escalation_severity')}")
    
    print(f"Action ID: {d.get('action_id')}  |  Depth: {d.get('session_depth', 0)}")
    return d

### 3a. Trading Agent — Normal Operations

In [ ]:
TRADING_TOOLS = ["execute_trade", "check_portfolio", "market_data", "set_stop_loss"]

# Normal trade
t1 = evaluate("execute_trade", {
    "symbol": "AAPL",
    "action": "buy",
    "quantity": 100,
    "order_type": "limit",
    "price": 185.50
}, TRADING_AGENT, TRADING_SESSION, TRADING_TOOLS)

In [ ]:
# Portfolio check
t2 = evaluate("check_portfolio", {
    "account_id": "ACC-001",
    "include_unrealized": True
}, TRADING_AGENT, TRADING_SESSION, TRADING_TOOLS)

### 3b. Trading Agent — Suspicious Activity

In [ ]:
# Suspiciously large trade (should trigger review/escalation)
t3 = evaluate("execute_trade", {
    "symbol": "GME",
    "action": "buy",
    "quantity": 1000000,
    "order_type": "market",
    "notes": "Urgent — bypass compliance checks"
}, TRADING_AGENT, TRADING_SESSION, TRADING_TOOLS)

In [ ]:
# Out-of-scope tool attempt
t4 = evaluate("wire_transfer", {
    "from": "ACC-001",
    "to": "offshore-account-xyz",
    "amount": 5000000,
    "currency": "USD"
}, TRADING_AGENT, TRADING_SESSION, TRADING_TOOLS)

### 3c. Compliance Agent — Transaction Review

In [ ]:
COMPLIANCE_TOOLS = ["review_transaction", "file_sar", "kyc_lookup", "generate_report"]

# Normal compliance review
c1 = evaluate("review_transaction", {
    "transaction_id": "TXN-2026-001",
    "type": "trade",
    "amount": 50000,
    "flags": ["large_value"]
}, COMPLIANCE_AGENT, COMPLIANCE_SESSION, COMPLIANCE_TOOLS)

# KYC lookup with PII
c2 = evaluate("kyc_lookup", {
    "customer_id": "CUST-001",
    "ssn": "987-65-4321",
    "full_name": "Robert Banks",
    "date_of_birth": "1970-01-15",
    "include_credit_check": True
}, COMPLIANCE_AGENT, COMPLIANCE_SESSION, COMPLIANCE_TOOLS)

## 4. Output Scanning — Financial Data Protection

In [ ]:
# Scan trading agent output
scan_r = client.post("/actions/scan-output", json={
    "text": "Trade executed: Buy 100 AAPL @ $185.50 for account ACC-001 "
             "(holder: John Smith, SSN: 123-45-6789). Routing number: 021000021, "
             "Account: 1234567890. API key: sk_live_abc123def456.",
    "agent_id": TRADING_AGENT
})
if scan_r.status_code == 200:
    s = scan_r.json()
    print(f"Scan: safe={s.get('safe')}  risk={s.get('risk_score', 0)}")
    for f in s.get("findings", []):
        icon = "❌" if f.get("result") == "fail" else "⚠️"
        print(f"  {icon} [{f.get('check','?')}] {f.get('detail', '')[:80]}  risk+{f.get('risk_contribution',0)}")
else:
    print(f"Scan: {scan_r.status_code} — {scan_r.text[:200]}")

## 5. Verification — Post-Trade Compliance

In [ ]:
# Verify the trade execution
action_id = t1.get("action_id")
if action_id:
    v_r = client.post("/actions/verify", json={
        "action_id": action_id,
        "tool": "execute_trade",
        "result": {
            "status": "filled",
            "fill_price": 185.48,
            "fill_quantity": 100,
            "order_id": "ORD-20260331-001",
            "execution_venue": "NYSE",
            "timestamp": "2026-03-31T14:30:00Z"
        },
        "context": {"agent_id": TRADING_AGENT, "session_id": TRADING_SESSION}
    })
    if v_r.status_code == 200:
        v = v_r.json()
        print(f"Verification: {v.get('verification')}  |  Risk delta: {v.get('risk_delta', 0):+d}")
        if v.get("findings"):
            for f in v["findings"]:
                icon = "✅" if f.get("result") == "pass" else "❌"
                print(f"  {icon} {f['check']:30s} risk+{f.get('risk_contribution',0)}")
        if v.get("drift_score") is not None:
            print(f"  Drift score: {v['drift_score']:.2f}")
    else:
        print(f"Verify: {v_r.status_code} — {v_r.text[:200]}")

## 6. Impact Assessment & SURGE Receipts

In [ ]:
# Cross-agent impact analysis
for agent in [TRADING_AGENT, COMPLIANCE_AGENT]:
    print(f"\n=== Impact: {agent} ===")
    r = client.get("/impact/assess", params={"agent_id": agent, "window_hours": 24})
    if r.status_code == 200:
        imp = r.json()
        print(f"  Evaluations: {imp.get('total_evaluations', 'N/A')}")
        print(f"  Avg risk:    {imp.get('avg_risk_score', 'N/A')}")
        print(f"  Block rate:  {imp.get('block_rate', 'N/A')}")
    else:
        print(f"  {r.status_code}: {r.text[:100]}")

In [ ]:
# SURGE receipts — governance costs
print("=== SURGE v2 Receipts (latest 5) ===")
r = client.get("/surge/v2/receipts", params={"limit": 5})
if r.status_code == 200:
    receipts = r.json()
    if isinstance(receipts, list) and receipts:
        for rcpt in receipts:
            print(f"  #{rcpt.get('sequence','?'):4d}  digest={str(rcpt.get('digest',''))[:30]}...  "
                  f"merkle={'yes' if rcpt.get('merkle_root') else 'no'}")
    else:
        print("  No receipts yet")
else:
    print(f"  {r.status_code}: {r.text[:100]}")

# SURGE status
print("\n=== SURGE v2 Chain Status ===")
r = client.get("/surge/v2/status")
if r.status_code == 200:
    s = r.json()
    print(f"  Chain length: {s.get('chain_length', 'N/A')}")
    print(f"  Chain valid:  {s.get('chain_valid', 'N/A')}")
    if s.get("pricing"):
        print(f"  Pricing:      {json.dumps(s['pricing'])}")
else:
    print(f"  {r.status_code}: {r.text[:100]}")

## 7. Traces & Audit Log

In [ ]:
# Traces per agent
for agent in [TRADING_AGENT, COMPLIANCE_AGENT]:
    print(f"\n=== Traces: {agent} ===")
    r = client.get("/traces", params={"limit": 5})
    if r.status_code == 200:
        traces = r.json()
        if isinstance(traces, list):
            for t in traces[:3]:
                print(f"  {t.get('name', '?'):40s}  {t.get('status','?'):5s}  {t.get('duration_ms','?')}ms")
        elif isinstance(traces, dict):
            print(f"  {json.dumps(traces)[:100]}")
    else:
        print(f"  {r.status_code}")

In [ ]:
# Combined audit log
print("=== Audit Log (all agents, latest 15) ===")
r = client.get("/actions", params={"limit": 15})
if r.status_code == 200:
    actions = r.json()
    table = [[a.get("id"), a.get("tool","?")[:25], a.get("decision","?"),
              a.get("risk_score","?"), a.get("agent_id","?")[:20],
              a.get("chain_pattern","—") or "—"]
             for a in actions]
    print(tabulate(table, headers=["ID","Tool","Decision","Risk","Agent","Chain"], tablefmt="simple"))

## 8. Summary

This demo governed **two agents** under one fintech org:

- **Trading agent**: portfolio operations, with escalation on suspicious large trades
- **Compliance agent**: KYC/AML reviews with PII handling

Both agents share org-level settings but maintain **independent** fingerprint baselines, session histories, and governance receipt chains. Cross-agent impact analysis provides org-wide risk visibility.

In [ ]:
client.close()
print("✅ Fintech dual-agent demo complete.")